# Temperature

In [7]:
# Land only

import xarray as xr
import numpy as np
import pandas as pd
import regionmask

# 📌 Settings
BASE_PATH = "/Users/dianindrawati/Documents/ERA5_datasets/Temperature/ERA5_tas_base_19712000_seasonal.nc"
PRES_PATH = "/Users/dianindrawati/Documents/ERA5_datasets/Temperature/ERA5_tas_present_19812010_seasonal.nc"
OUT_CSV = "/Users/dianindrawati/Documents/ERA5_datasets/Temperature/ERA5_tas_present_vs_baseline_with_std_land.csv"
VAR = "t2m"
# Daily files: 1 Dec 1970 – 30 Nov 2000 (baseline), 1 Dec 1980 – 30 Nov 2010 (present);
# lat −90→90 (ascending), lon 0–359. This gives 30 complete seasons per period:
# DJF 1971–2000 / 1981–2010, and MAM/JJA/SON 1971–2000 / 1981–2010
# (1970/1980 contain only December, so the 1970/1980 start adds no extra year).
BASE_YEARS = {"DJF": (1971, 2000), "other": (1970, 2000)}
PRES_YEARS = {"DJF": (1981, 2010), "other": (1980, 2010)}
SEASONS = ["DJF", "MAM", "JJA", "SON"]

# --- Regions ---
regions = {
    "Global": {"lat_min": -60, "lat_max": 60},
    "Tropics": {"lat_min": -23.5, "lat_max": 23.5},
    "Subtropics_N": {"lat_min": 23.5, "lat_max": 35},
    "Subtropics_S": {"lat_min": -35, "lat_max": -23.5},
    "Mid_Latitudes_N": {"lat_min": 35, "lat_max": 60},
    "Mid_Latitudes_S": {"lat_min": -60, "lat_max": -35},
}

SEASON_MONTHS = {"DJF": [12, 1, 2], "MAM": [3, 4, 5], "JJA": [6, 7, 8], "SON": [9, 10, 11]}

# --- Seasonal extraction ---
# DJF is labelled by the January year (Dec 1980 + Jan/Feb 1981 → 1981).
# Seasons missing any of their 3 months are dropped as a safeguard so every
# season is a full 3-month mean (with Dec 1970/1980 in the files, DJF 1971/1981
# is complete and kept).
def extract_season(ds, season):
    time = ds["valid_time"]
    month = time.dt.month
    year = time.dt.year
    season_year = xr.where(month == 12, year + 1, year) if season == "DJF" else year

    ds = ds.assign_coords(season_year=("valid_time", season_year.values))
    ds_season = ds.sel(valid_time=month.isin(SEASON_MONTHS[season]))

    # keep only season-years with all 3 months present
    sy = ds_season["season_year"].values
    mo = ds_season["valid_time"].dt.month.values
    n_months = pd.Series(mo).groupby(sy).nunique()
    complete = n_months[n_months == 3].index.values
    ds_season = ds_season.sel(valid_time=np.isin(sy, complete))

    grouped = ds_season.groupby("season_year").mean(dim="valid_time", skipna=True)
    return grouped.rename({"season_year": "year"})

# --- Load datasets ---
ds_baseline = xr.open_dataset(BASE_PATH)
ds_present = xr.open_dataset(PRES_PATH)

# --- Land mask (Natural Earth 110m, same as annual + CMIP6 analysis) ---
land_regions = regionmask.defined_regions.natural_earth_v5_0_0.land_110
land_mask = land_regions.mask(ds_baseline["lon"], ds_baseline["lat"])
is_land = land_mask.notnull()  # boolean (lat, lon)

# --- Latitude direction ---
latitudes = ds_baseline["lat"].values
lat_descending = latitudes[0] > latitudes[-1]

def sel_lat(da, lat_min, lat_max):
    return da.sel(lat=slice(lat_max, lat_min) if lat_descending else slice(lat_min, lat_max))

# --- Land-only area-weighted mean ---
# Weights = cos(lat) on land, 0 over ocean, applied jointly over (lat, lon)
# so the denominator only counts land area.
def weighted_mean_land(data, land):
    w = np.cos(np.deg2rad(data["lat"])) * land.astype(float)
    w = w.broadcast_like(data.isel(year=0, drop=True))
    return data.weighted(w).mean(dim=("lat", "lon"), skipna=True)

# --- Denominator check ---
# Recomputes the mean by hand as Σ(x·w) / Σ(w over valid land cells) and checks:
#   1) no ocean cell carries any weight,
#   2) no ocean cell has data (so nothing from the ocean reaches the numerator),
#   3) the hand-computed land-only mean equals weighted_mean_land.
# Raises an error if any check fails.
def check_land_denominator(data, land, result, label):
    w = (np.cos(np.deg2rad(data["lat"])) * land.astype(float)).broadcast_like(data.isel(year=0, drop=True))
    valid = data.notnull()
    ocean_weight = float(w.where(~land).sum())
    ocean_valid = int((valid & ~land).sum())
    denom = w.where(valid).sum(dim=("lat", "lon"))                    # land-only Σw per year
    numer = (data * w).sum(dim=("lat", "lon"), skipna=True)
    manual = numer / denom
    max_diff = float(abs(manual - result).max())
    full_band_w = float(np.cos(np.deg2rad(data["lat"])).broadcast_like(w).sum())
    land_frac = float(denom.mean()) / full_band_w                      # share of band area used

    assert ocean_weight == 0.0, f"{label}: ocean cells have weight {ocean_weight}"
    assert ocean_valid == 0, f"{label}: {ocean_valid} ocean cell-years have data"
    assert max_diff < 1e-6, f"{label}: manual vs weighted mean differ by {max_diff}"
    return land_frac

denom_log = []

# --- Compute ---
results = {region: {} for region in regions}

for season in SEASONS:
    key = "DJF" if season == "DJF" else "other"
    base = extract_season(ds_baseline, season).sel(year=slice(*BASE_YEARS[key]))
    present = extract_season(ds_present, season).sel(year=slice(*PRES_YEARS[key]))

    base_land = base[VAR].where(is_land)
    present_land = present[VAR].where(is_land)

    for region, b in regions.items():
        base_reg = sel_lat(base_land, b["lat_min"], b["lat_max"])
        present_reg = sel_lat(present_land, b["lat_min"], b["lat_max"])
        land_reg = sel_lat(is_land, b["lat_min"], b["lat_max"])

        # Spatial land-weighted means per year (K)
        base_k = weighted_mean_land(base_reg, land_reg)
        present_k = weighted_mean_land(present_reg, land_reg)

        # Verify the denominator is land-only
        lf_b = check_land_denominator(base_reg, land_reg, base_k, f"{region} {season} baseline")
        lf_p = check_land_denominator(present_reg, land_reg, present_k, f"{region} {season} present")
        denom_log.append((region, season, lf_b, lf_p))

        # K → °C
        base_c = base_k - 273.15
        present_c = present_k - 273.15

        base_mean = base_c.mean("year").item()
        present_mean = present_c.mean("year").item()
        base_std = base_c.std("year").item()
        present_std = present_c.std("year").item()

        n_base = base_c.sizes["year"]
        n_present = present_c.sizes["year"]
        pooled_std = np.sqrt(((n_base - 1) * base_std**2 + (n_present - 1) * present_std**2)
                             / (n_base + n_present - 2))

        diff = present_mean - base_mean
        base_K = base_mean + 273.15  # absolute temperature for percentages

        results[region][season] = {
            "diff_C": diff,
            "pct_change": diff / base_K * 100,
            "std_dev_C": pooled_std,
            "std_dev_percent": pooled_std / base_K * 100,
            "base_mean_C": base_mean,
            "present_mean_C": present_mean,
            "n_base": n_base,
            "n_present": n_present,
            "n_land_cells": int(land_reg.sum()),
        }

# --- Print denominator verification ---
print("\nDenominator check (passed for every region/season/period):")
print("  ocean weight = 0, ocean data = 0, manual Σ(x·w)/Σ_land(w) == weighted mean")
print(f"{'Region':<20}{'Season':<8}{'Land area frac (base)':>24}{'(present)':>12}")
for region, season, lf_b, lf_p in denom_log:
    print(f"{region:<20}{season:<8}{lf_b:>24.3f}{lf_p:>12.3f}")

# --- Print table ---
print("\nLand-Only Seasonal Temperature Changes (Present - Baseline) with Standard Deviation (cos-lat weighted, land cells only):")
print(f"{'Region':<20}{'Season':<8}{'Diff (°C)':>10}{'Change (%)':>12}{'Std Dev (°C)':>14}{'Std Dev (%)':>12}{'Land cells':>12}")
print("-" * 88)
for region in results:
    for season in SEASONS:
        r = results[region][season]
        print(f"{region:<20}{season:<8}{r['diff_C']:>10.3f}{r['pct_change']:>12.2f}"
              f"{r['std_dev_C']:>14.3f}{r['std_dev_percent']:>12.2f}{r['n_land_cells']:>12}")

# --- Detailed statistics ---
print("\n" + "=" * 100)
print("Detailed Land-Only Statistics by Period:")
print("=" * 100)
for region in results:
    print(f"\n{region}:")
    for season in SEASONS:
        r = results[region][season]
        print(f"  {season}: Baseline = {r['base_mean_C']:.3f} °C (n={r['n_base']}), "
              f"Present = {r['present_mean_C']:.3f} °C (n={r['n_present']}), "
              f"Diff = {r['diff_C']:.3f} °C, Pooled Std Dev = {r['std_dev_C']:.3f} °C")

# --- Save CSV ---
records = [
    {
        "Region": region,
        "Season": season,
        "Diff_C": m["diff_C"],
        "Pct_Change": m["pct_change"],
        "Std_Dev_C": m["std_dev_C"],
        "Std_Dev_Percent": m["std_dev_percent"],
        "Base_Mean_C": m["base_mean_C"],
        "Present_Mean_C": m["present_mean_C"],
        "N_Base_Years": m["n_base"],
        "N_Present_Years": m["n_present"],
        "N_Land_Cells": m["n_land_cells"],
    }
    for region, seasons_dict in results.items()
    for season, m in seasons_dict.items()
]
pd.DataFrame(records).to_csv(OUT_CSV, index=False)
print(f"\nResults saved to: {OUT_CSV}")

ds_baseline.close()
ds_present.close()


Denominator check (passed for every region/season/period):
  ocean weight = 0, ocean data = 0, manual Σ(x·w)/Σ_land(w) == weighted mean
Region              Season     Land area frac (base)   (present)
Global              DJF                        0.267       0.267
Tropics             DJF                        0.246       0.246
Subtropics_N        DJF                        0.409       0.409
Subtropics_S        DJF                        0.191       0.191
Mid_Latitudes_N     DJF                        0.515       0.515
Mid_Latitudes_S     DJF                        0.034       0.034
Global              MAM                        0.267       0.267
Tropics             MAM                        0.246       0.246
Subtropics_N        MAM                        0.409       0.409
Subtropics_S        MAM                        0.191       0.191
Mid_Latitudes_N     MAM                        0.515       0.515
Mid_Latitudes_S     MAM                        0.034       0.034
Global            

# Precipitable Water

In [8]:
# Land only

import xarray as xr
import numpy as np
import pandas as pd
import regionmask

# 📌 Settings
BASE_PATH = "/Users/dianindrawati/Documents/ERA5_datasets/Precipitable_Water/ERA5_prw_base_19712000_seasonal.nc"
PRES_PATH = "/Users/dianindrawati/Documents/ERA5_datasets/Precipitable_Water/ERA5_prw_present_19812010_seasonal.nc"
OUT_CSV = "/Users/dianindrawati/Documents/ERA5_datasets/Precipitable_Water/ERA5_prw_present_vs_baseline_with_std_land.csv"
VAR = "tcwv"   # total column water vapour (kg m-2 ≡ mm)
# Daily files: 1 Dec 1970 – 30 Nov 2000 (baseline), 1 Dec 1980 – 30 Nov 2010 (present);
# lat −90→90 (ascending), lon 0–359. This gives 30 complete seasons per period:
# DJF 1971–2000 / 1981–2010, and MAM/JJA/SON 1971–2000 / 1981–2010
# (1970/1980 contain only December, so the 1970/1980 start adds no extra year).
BASE_YEARS = {"DJF": (1971, 2000), "other": (1970, 2000)}
PRES_YEARS = {"DJF": (1981, 2010), "other": (1980, 2010)}
SEASONS = ["DJF", "MAM", "JJA", "SON"]

# --- Regions ---
regions = {
    "Global": {"lat_min": -60, "lat_max": 60},
    "Tropics": {"lat_min": -23.5, "lat_max": 23.5},
    "Subtropics_N": {"lat_min": 23.5, "lat_max": 35},
    "Subtropics_S": {"lat_min": -35, "lat_max": -23.5},
    "Mid_Latitudes_N": {"lat_min": 35, "lat_max": 60},
    "Mid_Latitudes_S": {"lat_min": -60, "lat_max": -35},
}

SEASON_MONTHS = {"DJF": [12, 1, 2], "MAM": [3, 4, 5], "JJA": [6, 7, 8], "SON": [9, 10, 11]}

# --- Seasonal extraction ---
# DJF is labelled by the January year (Dec 1980 + Jan/Feb 1981 → 1981).
# Seasons missing any of their 3 months are dropped as a safeguard so every
# season is a full 3-month mean (with Dec 1970/1980 in the files, DJF 1971/1981
# is complete and kept).
def extract_season(ds, season):
    time = ds["valid_time"]
    month = time.dt.month
    year = time.dt.year
    season_year = xr.where(month == 12, year + 1, year) if season == "DJF" else year

    ds = ds.assign_coords(season_year=("valid_time", season_year.values))
    ds_season = ds.sel(valid_time=month.isin(SEASON_MONTHS[season]))

    # keep only season-years with all 3 months present
    sy = ds_season["season_year"].values
    mo = ds_season["valid_time"].dt.month.values
    n_months = pd.Series(mo).groupby(sy).nunique()
    complete = n_months[n_months == 3].index.values
    ds_season = ds_season.sel(valid_time=np.isin(sy, complete))

    grouped = ds_season.groupby("season_year").mean(dim="valid_time", skipna=True)
    return grouped.rename({"season_year": "year"})

# --- Load datasets ---
ds_baseline = xr.open_dataset(BASE_PATH)
ds_present = xr.open_dataset(PRES_PATH)

# --- Land mask (Natural Earth 110m, same as annual + CMIP6 analysis) ---
land_regions = regionmask.defined_regions.natural_earth_v5_0_0.land_110
land_mask = land_regions.mask(ds_baseline["lon"], ds_baseline["lat"])
is_land = land_mask.notnull()  # boolean (lat, lon)

# --- Latitude direction ---
latitudes = ds_baseline["lat"].values
lat_descending = latitudes[0] > latitudes[-1]

def sel_lat(da, lat_min, lat_max):
    return da.sel(lat=slice(lat_max, lat_min) if lat_descending else slice(lat_min, lat_max))

# --- Land-only area-weighted mean ---
# Weights = cos(lat) on land, 0 over ocean, applied jointly over (lat, lon)
# so the denominator only counts land area.
def weighted_mean_land(data, land):
    w = np.cos(np.deg2rad(data["lat"])) * land.astype(float)
    w = w.broadcast_like(data.isel(year=0, drop=True))
    return data.weighted(w).mean(dim=("lat", "lon"), skipna=True)

# --- Denominator check ---
# Recomputes the mean by hand as Σ(x·w) / Σ(w over valid land cells) and checks:
#   1) no ocean cell carries any weight,
#   2) no ocean cell has data (so nothing from the ocean reaches the numerator),
#   3) the hand-computed land-only mean equals weighted_mean_land.
# Raises an error if any check fails.
def check_land_denominator(data, land, result, label):
    w = (np.cos(np.deg2rad(data["lat"])) * land.astype(float)).broadcast_like(data.isel(year=0, drop=True))
    valid = data.notnull()
    ocean_weight = float(w.where(~land).sum())
    ocean_valid = int((valid & ~land).sum())
    denom = w.where(valid).sum(dim=("lat", "lon"))                    # land-only Σw per year
    numer = (data * w).sum(dim=("lat", "lon"), skipna=True)
    manual = numer / denom
    max_diff = float(abs(manual - result).max())
    full_band_w = float(np.cos(np.deg2rad(data["lat"])).broadcast_like(w).sum())
    land_frac = float(denom.mean()) / full_band_w                      # share of band area used

    assert ocean_weight == 0.0, f"{label}: ocean cells have weight {ocean_weight}"
    assert ocean_valid == 0, f"{label}: {ocean_valid} ocean cell-years have data"
    assert max_diff < 1e-6, f"{label}: manual vs weighted mean differ by {max_diff}"
    return land_frac

denom_log = []

# --- Compute ---
results = {region: {} for region in regions}

for season in SEASONS:
    key = "DJF" if season == "DJF" else "other"
    base = extract_season(ds_baseline, season).sel(year=slice(*BASE_YEARS[key]))
    present = extract_season(ds_present, season).sel(year=slice(*PRES_YEARS[key]))

    base_land = base[VAR].where(is_land)
    present_land = present[VAR].where(is_land)

    for region, b in regions.items():
        base_reg = sel_lat(base_land, b["lat_min"], b["lat_max"])
        present_reg = sel_lat(present_land, b["lat_min"], b["lat_max"])
        land_reg = sel_lat(is_land, b["lat_min"], b["lat_max"])

        # Spatial land-weighted means per year (mm)
        base_sm = weighted_mean_land(base_reg, land_reg)
        present_sm = weighted_mean_land(present_reg, land_reg)

        # Verify the denominator is land-only
        lf_b = check_land_denominator(base_reg, land_reg, base_sm, f"{region} {season} baseline")
        lf_p = check_land_denominator(present_reg, land_reg, present_sm, f"{region} {season} present")
        denom_log.append((region, season, lf_b, lf_p))

        base_mean = base_sm.mean("year").item()
        present_mean = present_sm.mean("year").item()
        base_std = base_sm.std("year").item()
        present_std = present_sm.std("year").item()

        n_base = base_sm.sizes["year"]
        n_present = present_sm.sizes["year"]
        pooled_std = np.sqrt(((n_base - 1) * base_std**2 + (n_present - 1) * present_std**2)
                             / (n_base + n_present - 2))

        diff = present_mean - base_mean

        results[region][season] = {
            "diff_mm": diff,
            "pct_change": diff / base_mean * 100,          # relative to baseline land mean
            "std_dev_mm": pooled_std,
            "std_dev_percent": pooled_std / base_mean * 100,
            "base_mean_mm": base_mean,
            "present_mean_mm": present_mean,
            "n_base": n_base,
            "n_present": n_present,
            "n_land_cells": int(land_reg.sum()),
        }

# --- Print denominator verification ---
print("\nDenominator check (passed for every region/season/period):")
print("  ocean weight = 0, ocean data = 0, manual Σ(x·w)/Σ_land(w) == weighted mean")
print(f"{'Region':<20}{'Season':<8}{'Land area frac (base)':>24}{'(present)':>12}")
for region, season, lf_b, lf_p in denom_log:
    print(f"{region:<20}{season:<8}{lf_b:>24.3f}{lf_p:>12.3f}")

# --- Print table ---
print("\nLand-Only Seasonal Precipitable Water Changes (Present - Baseline) with Standard Deviation (cos-lat weighted, land cells only):")
print(f"{'Region':<20}{'Season':<8}{'Diff (mm)':>10}{'Change (%)':>12}{'Std Dev (mm)':>14}{'Std Dev (%)':>12}{'Land cells':>12}")
print("-" * 88)
for region in results:
    for season in SEASONS:
        r = results[region][season]
        print(f"{region:<20}{season:<8}{r['diff_mm']:>10.3f}{r['pct_change']:>12.2f}"
              f"{r['std_dev_mm']:>14.3f}{r['std_dev_percent']:>12.2f}{r['n_land_cells']:>12}")

# --- Detailed statistics ---
print("\n" + "=" * 100)
print("Detailed Land-Only Statistics by Period:")
print("=" * 100)
for region in results:
    print(f"\n{region}:")
    for season in SEASONS:
        r = results[region][season]
        print(f"  {season}: Baseline = {r['base_mean_mm']:.3f} mm (n={r['n_base']}), "
              f"Present = {r['present_mean_mm']:.3f} mm (n={r['n_present']}), "
              f"Diff = {r['diff_mm']:.3f} mm, Pooled Std Dev = {r['std_dev_mm']:.3f} mm")

# --- Save CSV ---
records = [
    {
        "Region": region,
        "Season": season,
        "Diff_mm": m["diff_mm"],
        "Pct_Change": m["pct_change"],
        "Std_Dev_mm": m["std_dev_mm"],
        "Std_Dev_Percent": m["std_dev_percent"],
        "Base_Mean_mm": m["base_mean_mm"],
        "Present_Mean_mm": m["present_mean_mm"],
        "N_Base_Years": m["n_base"],
        "N_Present_Years": m["n_present"],
        "N_Land_Cells": m["n_land_cells"],
    }
    for region, seasons_dict in results.items()
    for season, m in seasons_dict.items()
]
pd.DataFrame(records).to_csv(OUT_CSV, index=False)
print(f"\nResults saved to: {OUT_CSV}")

ds_baseline.close()
ds_present.close()


Denominator check (passed for every region/season/period):
  ocean weight = 0, ocean data = 0, manual Σ(x·w)/Σ_land(w) == weighted mean
Region              Season     Land area frac (base)   (present)
Global              DJF                        0.267       0.267
Tropics             DJF                        0.246       0.246
Subtropics_N        DJF                        0.409       0.409
Subtropics_S        DJF                        0.191       0.191
Mid_Latitudes_N     DJF                        0.515       0.515
Mid_Latitudes_S     DJF                        0.034       0.034
Global              MAM                        0.267       0.267
Tropics             MAM                        0.246       0.246
Subtropics_N        MAM                        0.409       0.409
Subtropics_S        MAM                        0.191       0.191
Mid_Latitudes_N     MAM                        0.515       0.515
Mid_Latitudes_S     MAM                        0.034       0.034
Global            

# Precipitation

In [11]:
# Land only

import xarray as xr
import numpy as np
import pandas as pd
import regionmask

# 📌 Settings
BASE_PATH = "/Users/dianindrawati/Documents/ERA5_datasets/Precipitation/ERA5_pr_base_19712000_seasonal.nc"
PRES_PATH = "/Users/dianindrawati/Documents/ERA5_datasets/Precipitation/ERA5_pr_present_19812010_seasonal.nc"
OUT_CSV = "/Users/dianindrawati/Documents/ERA5_datasets/Precipitation/ERA5_pr_present_vs_baseline_with_std_land.csv"
VAR = "tp"     # total precipitation (m/day → converted to mm/day)
SCALE = 1000   # m/day → mm/day
# Daily files: 1 Dec 1970 – 30 Nov 2000 (baseline), 1 Dec 1980 – 30 Nov 2010 (present);
# lat −90→90 (ascending), lon 0–359. This gives 30 complete seasons per period:
# DJF 1971–2000 / 1981–2010, and MAM/JJA/SON 1971–2000 / 1981–2010
# (1970/1980 contain only December, so the 1970/1980 start adds no extra year).
BASE_YEARS = {"DJF": (1971, 2000), "other": (1970, 2000)}
PRES_YEARS = {"DJF": (1981, 2010), "other": (1980, 2010)}
SEASONS = ["DJF", "MAM", "JJA", "SON"]

# --- Regions ---
regions = {
    "Global": {"lat_min": -60, "lat_max": 60},
    "Tropics": {"lat_min": -23.5, "lat_max": 23.5},
    "Subtropics_N": {"lat_min": 23.5, "lat_max": 35},
    "Subtropics_S": {"lat_min": -35, "lat_max": -23.5},
    "Mid_Latitudes_N": {"lat_min": 35, "lat_max": 60},
    "Mid_Latitudes_S": {"lat_min": -60, "lat_max": -35},
}

SEASON_MONTHS = {"DJF": [12, 1, 2], "MAM": [3, 4, 5], "JJA": [6, 7, 8], "SON": [9, 10, 11]}

# --- Seasonal extraction ---
# DJF is labelled by the January year (Dec 1980 + Jan/Feb 1981 → 1981).
# Seasons missing any of their 3 months are dropped as a safeguard so every
# season is a full 3-month mean (with Dec 1970/1980 in the files, DJF 1971/1981
# is complete and kept).
def extract_season(ds, season):
    time = ds["valid_time"]
    month = time.dt.month
    year = time.dt.year
    season_year = xr.where(month == 12, year + 1, year) if season == "DJF" else year

    ds = ds.assign_coords(season_year=("valid_time", season_year.values))
    ds_season = ds.sel(valid_time=month.isin(SEASON_MONTHS[season]))

    # keep only season-years with all 3 months present
    sy = ds_season["season_year"].values
    mo = ds_season["valid_time"].dt.month.values
    n_months = pd.Series(mo).groupby(sy).nunique()
    complete = n_months[n_months == 3].index.values
    ds_season = ds_season.sel(valid_time=np.isin(sy, complete))

    grouped = ds_season.groupby("season_year").mean(dim="valid_time", skipna=True)
    return grouped.rename({"season_year": "year"})

# --- Load datasets ---
ds_baseline = xr.open_dataset(BASE_PATH)
ds_present = xr.open_dataset(PRES_PATH)

# --- Land mask (Natural Earth 110m, same as annual + CMIP6 analysis) ---
land_regions = regionmask.defined_regions.natural_earth_v5_0_0.land_110
land_mask = land_regions.mask(ds_baseline["lon"], ds_baseline["lat"])
is_land = land_mask.notnull()  # boolean (lat, lon)

# --- Latitude direction ---
latitudes = ds_baseline["lat"].values
lat_descending = latitudes[0] > latitudes[-1]

def sel_lat(da, lat_min, lat_max):
    return da.sel(lat=slice(lat_max, lat_min) if lat_descending else slice(lat_min, lat_max))

# --- Land-only area-weighted mean ---
# Weights = cos(lat) on land, 0 over ocean, applied jointly over (lat, lon)
# so the denominator only counts land area.
def weighted_mean_land(data, land):
    w = np.cos(np.deg2rad(data["lat"])) * land.astype(float)
    w = w.broadcast_like(data.isel(year=0, drop=True))
    return data.weighted(w).mean(dim=("lat", "lon"), skipna=True)

# --- Denominator check ---
# Recomputes the mean by hand as Σ(x·w) / Σ(w over valid land cells) and checks:
#   1) no ocean cell carries any weight,
#   2) no ocean cell has data (so nothing from the ocean reaches the numerator),
#   3) the hand-computed land-only mean equals weighted_mean_land.
# Raises an error if any check fails.
def check_land_denominator(data, land, result, label):
    w = (np.cos(np.deg2rad(data["lat"])) * land.astype(float)).broadcast_like(data.isel(year=0, drop=True))
    valid = data.notnull()
    ocean_weight = float(w.where(~land).sum())
    ocean_valid = int((valid & ~land).sum())
    denom = w.where(valid).sum(dim=("lat", "lon"))                    # land-only Σw per year
    numer = (data * w).sum(dim=("lat", "lon"), skipna=True)
    manual = numer / denom
    max_diff = float(abs(manual - result).max())
    full_band_w = float(np.cos(np.deg2rad(data["lat"])).broadcast_like(w).sum())
    land_frac = float(denom.mean()) / full_band_w                      # share of band area used

    assert ocean_weight == 0.0, f"{label}: ocean cells have weight {ocean_weight}"
    assert ocean_valid == 0, f"{label}: {ocean_valid} ocean cell-years have data"
    assert max_diff < 1e-6, f"{label}: manual vs weighted mean differ by {max_diff}"
    return land_frac

denom_log = []

# --- Compute ---
results = {region: {} for region in regions}

for season in SEASONS:
    key = "DJF" if season == "DJF" else "other"
    base = extract_season(ds_baseline, season).sel(year=slice(*BASE_YEARS[key]))
    present = extract_season(ds_present, season).sel(year=slice(*PRES_YEARS[key]))

    # Unit conversion m/day → mm/day, then land mask
    base_land = (base[VAR] * SCALE).where(is_land)
    present_land = (present[VAR] * SCALE).where(is_land)

    for region, b in regions.items():
        base_reg = sel_lat(base_land, b["lat_min"], b["lat_max"])
        present_reg = sel_lat(present_land, b["lat_min"], b["lat_max"])
        land_reg = sel_lat(is_land, b["lat_min"], b["lat_max"])

        # Spatial land-weighted means per year (mm/day)
        base_sm = weighted_mean_land(base_reg, land_reg)
        present_sm = weighted_mean_land(present_reg, land_reg)

        # Verify the denominator is land-only
        lf_b = check_land_denominator(base_reg, land_reg, base_sm, f"{region} {season} baseline")
        lf_p = check_land_denominator(present_reg, land_reg, present_sm, f"{region} {season} present")
        denom_log.append((region, season, lf_b, lf_p))

        base_mean = base_sm.mean("year").item()
        present_mean = present_sm.mean("year").item()
        base_std = base_sm.std("year").item()
        present_std = present_sm.std("year").item()

        n_base = base_sm.sizes["year"]
        n_present = present_sm.sizes["year"]
        pooled_std = np.sqrt(((n_base - 1) * base_std**2 + (n_present - 1) * present_std**2)
                             / (n_base + n_present - 2))

        diff = present_mean - base_mean

        results[region][season] = {
            "diff_mm_day": diff,
            "pct_change": diff / base_mean * 100,          # relative to baseline land mean
            "std_dev_mm_day": pooled_std,
            "std_dev_percent": pooled_std / base_mean * 100,
            "base_mean_mm_day": base_mean,
            "present_mean_mm_day": present_mean,
            "n_base": n_base,
            "n_present": n_present,
            "n_land_cells": int(land_reg.sum()),
        }

# --- Print denominator verification ---
print("\nDenominator check (passed for every region/season/period):")
print("  ocean weight = 0, ocean data = 0, manual Σ(x·w)/Σ_land(w) == weighted mean")
print(f"{'Region':<20}{'Season':<8}{'Land area frac (base)':>24}{'(present)':>12}")
for region, season, lf_b, lf_p in denom_log:
    print(f"{region:<20}{season:<8}{lf_b:>24.3f}{lf_p:>12.3f}")

# --- Print table ---
print("\nLand-Only Seasonal Precipitation Changes (Present - Baseline) with Standard Deviation (cos-lat weighted, land cells only):")
print(f"{'Region':<20}{'Season':<8}{'Diff (mm/day)':>14}{'Change (%)':>12}{'Std Dev (mm/day)':>18}{'Std Dev (%)':>12}{'Land cells':>12}")
print("-" * 96)
for region in results:
    for season in SEASONS:
        r = results[region][season]
        print(f"{region:<20}{season:<8}{r['diff_mm_day']:>14.6f}{r['pct_change']:>12.2f}"
              f"{r['std_dev_mm_day']:>18.6f}{r['std_dev_percent']:>12.2f}{r['n_land_cells']:>12}")

# --- Detailed statistics ---
print("\n" + "=" * 100)
print("Detailed Land-Only Statistics by Period:")
print("=" * 100)
for region in results:
    print(f"\n{region}:")
    for season in SEASONS:
        r = results[region][season]
        print(f"  {season}: Baseline = {r['base_mean_mm_day']:.6f} mm/day (n={r['n_base']}), "
              f"Present = {r['present_mean_mm_day']:.6f} mm/day (n={r['n_present']}), "
              f"Diff = {r['diff_mm_day']:.6f} mm/day, Pooled Std Dev = {r['std_dev_mm_day']:.6f} mm/day")

# --- Save CSV ---
records = [
    {
        "Region": region,
        "Season": season,
        "Diff_mm_day": m["diff_mm_day"],
        "Pct_Change": m["pct_change"],
        "Std_Dev_mm_day": m["std_dev_mm_day"],
        "Std_Dev_Percent": m["std_dev_percent"],
        "Base_Mean_mm_day": m["base_mean_mm_day"],
        "Present_Mean_mm_day": m["present_mean_mm_day"],
        "N_Base_Years": m["n_base"],
        "N_Present_Years": m["n_present"],
        "N_Land_Cells": m["n_land_cells"],
    }
    for region, seasons_dict in results.items()
    for season, m in seasons_dict.items()
]
pd.DataFrame(records).to_csv(OUT_CSV, index=False)
print(f"\nResults saved to: {OUT_CSV}")

ds_baseline.close()
ds_present.close()


Denominator check (passed for every region/season/period):
  ocean weight = 0, ocean data = 0, manual Σ(x·w)/Σ_land(w) == weighted mean
Region              Season     Land area frac (base)   (present)
Global              DJF                        0.267       0.267
Tropics             DJF                        0.246       0.246
Subtropics_N        DJF                        0.409       0.409
Subtropics_S        DJF                        0.191       0.191
Mid_Latitudes_N     DJF                        0.515       0.515
Mid_Latitudes_S     DJF                        0.034       0.034
Global              MAM                        0.267       0.267
Tropics             MAM                        0.246       0.246
Subtropics_N        MAM                        0.409       0.409
Subtropics_S        MAM                        0.191       0.191
Mid_Latitudes_N     MAM                        0.515       0.515
Mid_Latitudes_S     MAM                        0.034       0.034
Global            

# Evapotranspiration

In [14]:
# Land only

import xarray as xr
import numpy as np
import pandas as pd
import regionmask

# 📌 Settings
BASE_PATH = "/Users/dianindrawati/Documents/ERA5_datasets/Evaporation/ERA5_e_base_19712000_seasonal.nc"
PRES_PATH = "/Users/dianindrawati/Documents/ERA5_datasets/Evaporation/ERA5_e_present_19812010_seasonal.nc"
OUT_CSV = "/Users/dianindrawati/Documents/ERA5_datasets/Evaporation/ERA5_e_present_vs_baseline_with_std_land.csv"
VAR = "e"      # ERA5 evaporation (m/day water equivalent → converted to mm/day)
FLIP_SIGN = True  # ERA5 'e' is negative for evaporation (downward-positive convention);
                  # True makes ET positive, same as the annual ET script and CMIP6 hfls-derived ET
SCALE = 1000 * (-1 if FLIP_SIGN else 1)   # m/day → mm/day (with optional sign flip)
# Daily files: 1 Dec 1970 – 30 Nov 2000 (baseline), 1 Dec 1980 – 30 Nov 2010 (present);
# lat −90→90 (ascending), lon 0–359. This gives 30 complete seasons per period:
# DJF 1971–2000 / 1981–2010, and MAM/JJA/SON 1971–2000 / 1981–2010
# (1970/1980 contain only December, so the 1970/1980 start adds no extra year).
BASE_YEARS = {"DJF": (1971, 2000), "other": (1970, 2000)}
PRES_YEARS = {"DJF": (1981, 2010), "other": (1980, 2010)}
SEASONS = ["DJF", "MAM", "JJA", "SON"]

# --- Regions ---
regions = {
    "Global": {"lat_min": -60, "lat_max": 60},
    "Tropics": {"lat_min": -23.5, "lat_max": 23.5},
    "Subtropics_N": {"lat_min": 23.5, "lat_max": 35},
    "Subtropics_S": {"lat_min": -35, "lat_max": -23.5},
    "Mid_Latitudes_N": {"lat_min": 35, "lat_max": 60},
    "Mid_Latitudes_S": {"lat_min": -60, "lat_max": -35},
}

SEASON_MONTHS = {"DJF": [12, 1, 2], "MAM": [3, 4, 5], "JJA": [6, 7, 8], "SON": [9, 10, 11]}

# --- Seasonal extraction ---
# DJF is labelled by the January year (Dec 1980 + Jan/Feb 1981 → 1981).
# Seasons missing any of their 3 months are dropped as a safeguard so every
# season is a full 3-month mean (with Dec 1970/1980 in the files, DJF 1971/1981
# is complete and kept).
def extract_season(ds, season):
    time = ds["valid_time"]
    month = time.dt.month
    year = time.dt.year
    season_year = xr.where(month == 12, year + 1, year) if season == "DJF" else year

    ds = ds.assign_coords(season_year=("valid_time", season_year.values))
    ds_season = ds.sel(valid_time=month.isin(SEASON_MONTHS[season]))

    # keep only season-years with all 3 months present
    sy = ds_season["season_year"].values
    mo = ds_season["valid_time"].dt.month.values
    n_months = pd.Series(mo).groupby(sy).nunique()
    complete = n_months[n_months == 3].index.values
    ds_season = ds_season.sel(valid_time=np.isin(sy, complete))

    grouped = ds_season.groupby("season_year").mean(dim="valid_time", skipna=True)
    return grouped.rename({"season_year": "year"})

# --- Load datasets ---
ds_baseline = xr.open_dataset(BASE_PATH)
ds_present = xr.open_dataset(PRES_PATH)

# --- Land mask (Natural Earth 110m, same as annual + CMIP6 analysis) ---
land_regions = regionmask.defined_regions.natural_earth_v5_0_0.land_110
land_mask = land_regions.mask(ds_baseline["lon"], ds_baseline["lat"])
is_land = land_mask.notnull()  # boolean (lat, lon)

# --- Latitude direction ---
latitudes = ds_baseline["lat"].values
lat_descending = latitudes[0] > latitudes[-1]

def sel_lat(da, lat_min, lat_max):
    return da.sel(lat=slice(lat_max, lat_min) if lat_descending else slice(lat_min, lat_max))

# --- Land-only area-weighted mean ---
# Weights = cos(lat) on land, 0 over ocean, applied jointly over (lat, lon)
# so the denominator only counts land area.
def weighted_mean_land(data, land):
    w = np.cos(np.deg2rad(data["lat"])) * land.astype(float)
    w = w.broadcast_like(data.isel(year=0, drop=True))
    return data.weighted(w).mean(dim=("lat", "lon"), skipna=True)

# --- Denominator check ---
# Recomputes the mean by hand as Σ(x·w) / Σ(w over valid land cells) and checks:
#   1) no ocean cell carries any weight,
#   2) no ocean cell has data (so nothing from the ocean reaches the numerator),
#   3) the hand-computed land-only mean equals weighted_mean_land.
# Raises an error if any check fails.
def check_land_denominator(data, land, result, label):
    w = (np.cos(np.deg2rad(data["lat"])) * land.astype(float)).broadcast_like(data.isel(year=0, drop=True))
    valid = data.notnull()
    ocean_weight = float(w.where(~land).sum())
    ocean_valid = int((valid & ~land).sum())
    denom = w.where(valid).sum(dim=("lat", "lon"))                    # land-only Σw per year
    numer = (data * w).sum(dim=("lat", "lon"), skipna=True)
    manual = numer / denom
    max_diff = float(abs(manual - result).max())
    full_band_w = float(np.cos(np.deg2rad(data["lat"])).broadcast_like(w).sum())
    land_frac = float(denom.mean()) / full_band_w                      # share of band area used

    assert ocean_weight == 0.0, f"{label}: ocean cells have weight {ocean_weight}"
    assert ocean_valid == 0, f"{label}: {ocean_valid} ocean cell-years have data"
    assert max_diff < 1e-6, f"{label}: manual vs weighted mean differ by {max_diff}"
    return land_frac

denom_log = []

# --- Compute ---
results = {region: {} for region in regions}

for season in SEASONS:
    key = "DJF" if season == "DJF" else "other"
    base = extract_season(ds_baseline, season).sel(year=slice(*BASE_YEARS[key]))
    present = extract_season(ds_present, season).sel(year=slice(*PRES_YEARS[key]))

    # Unit conversion m/day → mm/day (+ sign flip), then land mask
    base_land = (base[VAR] * SCALE).where(is_land)
    present_land = (present[VAR] * SCALE).where(is_land)

    for region, b in regions.items():
        base_reg = sel_lat(base_land, b["lat_min"], b["lat_max"])
        present_reg = sel_lat(present_land, b["lat_min"], b["lat_max"])
        land_reg = sel_lat(is_land, b["lat_min"], b["lat_max"])

        # Spatial land-weighted means per year (mm/day)
        base_sm = weighted_mean_land(base_reg, land_reg)
        present_sm = weighted_mean_land(present_reg, land_reg)

        # Verify the denominator is land-only
        lf_b = check_land_denominator(base_reg, land_reg, base_sm, f"{region} {season} baseline")
        lf_p = check_land_denominator(present_reg, land_reg, present_sm, f"{region} {season} present")
        denom_log.append((region, season, lf_b, lf_p))

        base_mean = base_sm.mean("year").item()
        present_mean = present_sm.mean("year").item()
        base_std = base_sm.std("year").item()
        present_std = present_sm.std("year").item()

        n_base = base_sm.sizes["year"]
        n_present = present_sm.sizes["year"]
        pooled_std = np.sqrt(((n_base - 1) * base_std**2 + (n_present - 1) * present_std**2)
                             / (n_base + n_present - 2))

        diff = present_mean - base_mean

        results[region][season] = {
            "diff_mm_day": diff,
            "pct_change": diff / base_mean * 100,          # relative to baseline land mean
            "std_dev_mm_day": pooled_std,
            "std_dev_percent": pooled_std / abs(base_mean) * 100,
            "base_mean_mm_day": base_mean,
            "present_mean_mm_day": present_mean,
            "n_base": n_base,
            "n_present": n_present,
            "n_land_cells": int(land_reg.sum()),
        }

# --- Print denominator verification ---
print("\nDenominator check (passed for every region/season/period):")
print("  ocean weight = 0, ocean data = 0, manual Σ(x·w)/Σ_land(w) == weighted mean")
print(f"{'Region':<20}{'Season':<8}{'Land area frac (base)':>24}{'(present)':>12}")
for region, season, lf_b, lf_p in denom_log:
    print(f"{region:<20}{season:<8}{lf_b:>24.3f}{lf_p:>12.3f}")

# --- Print table ---
print(f"\nSign convention: {'positive = upward flux (ET)' if FLIP_SIGN else 'ERA5 native (negative = evaporation)'}")
print("\nLand-Only Seasonal Evapotranspiration (ET) Changes (Present - Baseline) with Standard Deviation (cos-lat weighted, land cells only):")
print(f"{'Region':<20}{'Season':<8}{'Diff (mm/day)':>14}{'Change (%)':>12}{'Std Dev (mm/day)':>18}{'Std Dev (%)':>12}{'Land cells':>12}")
print("-" * 96)
for region in results:
    for season in SEASONS:
        r = results[region][season]
        print(f"{region:<20}{season:<8}{r['diff_mm_day']:>14.6f}{r['pct_change']:>12.2f}"
              f"{r['std_dev_mm_day']:>18.6f}{r['std_dev_percent']:>12.2f}{r['n_land_cells']:>12}")

# --- Detailed statistics ---
print("\n" + "=" * 100)
print("Detailed Land-Only Statistics by Period:")
print("=" * 100)
for region in results:
    print(f"\n{region}:")
    for season in SEASONS:
        r = results[region][season]
        print(f"  {season}: Baseline = {r['base_mean_mm_day']:.6f} mm/day (n={r['n_base']}), "
              f"Present = {r['present_mean_mm_day']:.6f} mm/day (n={r['n_present']}), "
              f"Diff = {r['diff_mm_day']:.6f} mm/day, Pooled Std Dev = {r['std_dev_mm_day']:.6f} mm/day")

# --- Save CSV ---
records = [
    {
        "Region": region,
        "Season": season,
        "Diff_mm_day": m["diff_mm_day"],
        "Pct_Change": m["pct_change"],
        "Std_Dev_mm_day": m["std_dev_mm_day"],
        "Std_Dev_Percent": m["std_dev_percent"],
        "Base_Mean_mm_day": m["base_mean_mm_day"],
        "Present_Mean_mm_day": m["present_mean_mm_day"],
        "N_Base_Years": m["n_base"],
        "N_Present_Years": m["n_present"],
        "N_Land_Cells": m["n_land_cells"],
    }
    for region, seasons_dict in results.items()
    for season, m in seasons_dict.items()
]
pd.DataFrame(records).to_csv(OUT_CSV, index=False)
print(f"\nResults saved to: {OUT_CSV}")

ds_baseline.close()
ds_present.close()


Denominator check (passed for every region/season/period):
  ocean weight = 0, ocean data = 0, manual Σ(x·w)/Σ_land(w) == weighted mean
Region              Season     Land area frac (base)   (present)
Global              DJF                        0.267       0.267
Tropics             DJF                        0.246       0.246
Subtropics_N        DJF                        0.409       0.409
Subtropics_S        DJF                        0.191       0.191
Mid_Latitudes_N     DJF                        0.515       0.515
Mid_Latitudes_S     DJF                        0.034       0.034
Global              MAM                        0.267       0.267
Tropics             MAM                        0.246       0.246
Subtropics_N        MAM                        0.409       0.409
Subtropics_S        MAM                        0.191       0.191
Mid_Latitudes_N     MAM                        0.515       0.515
Mid_Latitudes_S     MAM                        0.034       0.034
Global            

# Soil Moisture

In [17]:
# land only

import xarray as xr
import numpy as np
import pandas as pd
import regionmask

# 📌 Settings
BASE_PATH = "/Users/dianindrawati/Documents/ERA5_datasets/Soil_Moisture/ERA5_SM_base_19712000_seasonal.nc"
PRES_PATH = "/Users/dianindrawati/Documents/ERA5_datasets/Soil_Moisture/ERA5_SM_present_19812010_seasonal.nc"
OUT_CSV = "/Users/dianindrawati/Documents/ERA5_datasets/Soil_Moisture/ERA5_SM_present_vs_baseline_with_std_land.csv"
VAR = "swvl1"  # volumetric soil water, layer 1 (m³/m³) — no unit conversion
# Daily files: 1 Dec 1970 – 30 Nov 2000 (baseline), 1 Dec 1980 – 30 Nov 2010 (present);
# lat −90→90 (ascending), lon 0–359. This gives 30 complete seasons per period:
# DJF 1971–2000 / 1981–2010, and MAM/JJA/SON 1971–2000 / 1981–2010
# (1970/1980 contain only December, so the 1970/1980 start adds no extra year).
BASE_YEARS = {"DJF": (1971, 2000), "other": (1970, 2000)}
PRES_YEARS = {"DJF": (1981, 2010), "other": (1980, 2010)}
SEASONS = ["DJF", "MAM", "JJA", "SON"]

# --- Regions ---
regions = {
    "Global": {"lat_min": -60, "lat_max": 60},
    "Tropics": {"lat_min": -23.5, "lat_max": 23.5},
    "Subtropics_N": {"lat_min": 23.5, "lat_max": 35},
    "Subtropics_S": {"lat_min": -35, "lat_max": -23.5},
    "Mid_Latitudes_N": {"lat_min": 35, "lat_max": 60},
    "Mid_Latitudes_S": {"lat_min": -60, "lat_max": -35},
}

SEASON_MONTHS = {"DJF": [12, 1, 2], "MAM": [3, 4, 5], "JJA": [6, 7, 8], "SON": [9, 10, 11]}

# --- Seasonal extraction ---
# DJF is labelled by the January year (Dec 1980 + Jan/Feb 1981 → 1981).
# Seasons missing any of their 3 months are dropped as a safeguard so every
# season is a full 3-month mean (with Dec 1970/1980 in the files, DJF 1971/1981
# is complete and kept).
def extract_season(ds, season):
    time = ds["valid_time"]
    month = time.dt.month
    year = time.dt.year
    season_year = xr.where(month == 12, year + 1, year) if season == "DJF" else year

    ds = ds.assign_coords(season_year=("valid_time", season_year.values))
    ds_season = ds.sel(valid_time=month.isin(SEASON_MONTHS[season]))

    # keep only season-years with all 3 months present
    sy = ds_season["season_year"].values
    mo = ds_season["valid_time"].dt.month.values
    n_months = pd.Series(mo).groupby(sy).nunique()
    complete = n_months[n_months == 3].index.values
    ds_season = ds_season.sel(valid_time=np.isin(sy, complete))

    grouped = ds_season.groupby("season_year").mean(dim="valid_time", skipna=True)
    return grouped.rename({"season_year": "year"})

# --- Load datasets ---
ds_baseline = xr.open_dataset(BASE_PATH)
ds_present = xr.open_dataset(PRES_PATH)

# --- Land mask (Natural Earth 110m, same as annual + CMIP6 analysis) ---
land_regions = regionmask.defined_regions.natural_earth_v5_0_0.land_110
land_mask = land_regions.mask(ds_baseline["lon"], ds_baseline["lat"])
is_land = land_mask.notnull()  # boolean (lat, lon)

# --- Latitude direction ---
latitudes = ds_baseline["lat"].values
lat_descending = latitudes[0] > latitudes[-1]

def sel_lat(da, lat_min, lat_max):
    return da.sel(lat=slice(lat_max, lat_min) if lat_descending else slice(lat_min, lat_max))

# --- Land-only area-weighted mean ---
# Weights = cos(lat) on land, 0 over ocean, applied jointly over (lat, lon)
# so the denominator only counts land area.
def weighted_mean_land(data, land):
    w = np.cos(np.deg2rad(data["lat"])) * land.astype(float)
    w = w.broadcast_like(data.isel(year=0, drop=True))
    return data.weighted(w).mean(dim=("lat", "lon"), skipna=True)

# --- Denominator check ---
# Recomputes the mean by hand as Σ(x·w) / Σ(w over valid land cells) and checks:
#   1) no ocean cell carries any weight,
#   2) no ocean cell has data (so nothing from the ocean reaches the numerator),
#   3) the hand-computed land-only mean equals weighted_mean_land.
# Raises an error if any check fails.
def check_land_denominator(data, land, result, label):
    w = (np.cos(np.deg2rad(data["lat"])) * land.astype(float)).broadcast_like(data.isel(year=0, drop=True))
    valid = data.notnull()
    ocean_weight = float(w.where(~land).sum())
    ocean_valid = int((valid & ~land).sum())
    denom = w.where(valid).sum(dim=("lat", "lon"))                    # land-only Σw per year
    numer = (data * w).sum(dim=("lat", "lon"), skipna=True)
    manual = numer / denom
    max_diff = float(abs(manual - result).max())
    full_band_w = float(np.cos(np.deg2rad(data["lat"])).broadcast_like(w).sum())
    land_frac = float(denom.mean()) / full_band_w                      # share of band area used

    assert ocean_weight == 0.0, f"{label}: ocean cells have weight {ocean_weight}"
    assert ocean_valid == 0, f"{label}: {ocean_valid} ocean cell-years have data"
    assert max_diff < 1e-6, f"{label}: manual vs weighted mean differ by {max_diff}"
    return land_frac

denom_log = []

# --- Compute ---
results = {region: {} for region in regions}

for season in SEASONS:
    key = "DJF" if season == "DJF" else "other"
    base = extract_season(ds_baseline, season).sel(year=slice(*BASE_YEARS[key]))
    present = extract_season(ds_present, season).sel(year=slice(*PRES_YEARS[key]))

    base_land = base[VAR].where(is_land)
    present_land = present[VAR].where(is_land)

    for region, b in regions.items():
        base_reg = sel_lat(base_land, b["lat_min"], b["lat_max"])
        present_reg = sel_lat(present_land, b["lat_min"], b["lat_max"])
        land_reg = sel_lat(is_land, b["lat_min"], b["lat_max"])

        # Spatial land-weighted means per year (m³/m³)
        base_sm = weighted_mean_land(base_reg, land_reg)
        present_sm = weighted_mean_land(present_reg, land_reg)

        # Verify the denominator is land-only
        lf_b = check_land_denominator(base_reg, land_reg, base_sm, f"{region} {season} baseline")
        lf_p = check_land_denominator(present_reg, land_reg, present_sm, f"{region} {season} present")
        denom_log.append((region, season, lf_b, lf_p))

        base_mean = base_sm.mean("year").item()
        present_mean = present_sm.mean("year").item()
        base_std = base_sm.std("year").item()
        present_std = present_sm.std("year").item()

        n_base = base_sm.sizes["year"]
        n_present = present_sm.sizes["year"]
        pooled_std = np.sqrt(((n_base - 1) * base_std**2 + (n_present - 1) * present_std**2)
                             / (n_base + n_present - 2))

        diff = present_mean - base_mean

        results[region][season] = {
            "diff_m3_m3": diff,
            "pct_change": diff / base_mean * 100,          # relative to baseline land mean
            "std_dev_m3_m3": pooled_std,
            "std_dev_percent": pooled_std / base_mean * 100,
            "base_mean_m3_m3": base_mean,
            "present_mean_m3_m3": present_mean,
            "n_base": n_base,
            "n_present": n_present,
            "n_land_cells": int(land_reg.sum()),
            "n_valid_cells": int(base_reg.isel(year=0).notnull().sum()),  # land cells with swvl1 data
        }

# --- Print denominator verification ---
print("\nDenominator check (passed for every region/season/period):")
print("  ocean weight = 0, ocean data = 0, manual Σ(x·w)/Σ_land(w) == weighted mean")
print(f"{'Region':<20}{'Season':<8}{'Land area frac (base)':>24}{'(present)':>12}")
for region, season, lf_b, lf_p in denom_log:
    print(f"{region:<20}{season:<8}{lf_b:>24.3f}{lf_p:>12.3f}")

# --- Print table ---
print("\nLand-Only Seasonal Soil Moisture Changes (Present - Baseline) with Standard Deviation (cos-lat weighted, land cells only):")
print(f"{'Region':<20}{'Season':<8}{'Diff (m³/m³)':>14}{'Change (%)':>12}{'Std Dev (m³/m³)':>18}{'Std Dev (%)':>12}{'Land cells (valid)':>22}")
print("-" * 106)
for region in results:
    for season in SEASONS:
        r = results[region][season]
        print(f"{region:<20}{season:<8}{r['diff_m3_m3']:>14.6f}{r['pct_change']:>12.2f}"
              f"{r['std_dev_m3_m3']:>18.6f}{r['std_dev_percent']:>12.2f}{(str(r['n_land_cells']) + ' (' + str(r['n_valid_cells']) + ')'):>22}")

# --- Detailed statistics ---
print("\n" + "=" * 100)
print("Detailed Land-Only Statistics by Period:")
print("=" * 100)
for region in results:
    print(f"\n{region}:")
    for season in SEASONS:
        r = results[region][season]
        print(f"  {season}: Baseline = {r['base_mean_m3_m3']:.6f} m³/m³ (n={r['n_base']}), "
              f"Present = {r['present_mean_m3_m3']:.6f} m³/m³ (n={r['n_present']}), "
              f"Diff = {r['diff_m3_m3']:.6f} m³/m³, Pooled Std Dev = {r['std_dev_m3_m3']:.6f} m³/m³")

# --- Save CSV ---
records = [
    {
        "Region": region,
        "Season": season,
        "Diff_m3_m3": m["diff_m3_m3"],
        "Pct_Change": m["pct_change"],
        "Std_Dev_m3_m3": m["std_dev_m3_m3"],
        "Std_Dev_Percent": m["std_dev_percent"],
        "Base_Mean_m3_m3": m["base_mean_m3_m3"],
        "Present_Mean_m3_m3": m["present_mean_m3_m3"],
        "N_Base_Years": m["n_base"],
        "N_Present_Years": m["n_present"],
        "N_Land_Cells": m["n_land_cells"],
        "N_Valid_Land_Cells": m["n_valid_cells"],
    }
    for region, seasons_dict in results.items()
    for season, m in seasons_dict.items()
]
pd.DataFrame(records).to_csv(OUT_CSV, index=False)
print(f"\nResults saved to: {OUT_CSV}")

ds_baseline.close()
ds_present.close()


Denominator check (passed for every region/season/period):
  ocean weight = 0, ocean data = 0, manual Σ(x·w)/Σ_land(w) == weighted mean
Region              Season     Land area frac (base)   (present)
Global              DJF                        0.267       0.267
Tropics             DJF                        0.246       0.246
Subtropics_N        DJF                        0.409       0.409
Subtropics_S        DJF                        0.191       0.191
Mid_Latitudes_N     DJF                        0.515       0.515
Mid_Latitudes_S     DJF                        0.034       0.034
Global              MAM                        0.267       0.267
Tropics             MAM                        0.246       0.246
Subtropics_N        MAM                        0.409       0.409
Subtropics_S        MAM                        0.191       0.191
Mid_Latitudes_N     MAM                        0.515       0.515
Mid_Latitudes_S     MAM                        0.034       0.034
Global            

# Runoff

In [20]:
# Land only

import xarray as xr
import numpy as np
import pandas as pd
import regionmask

# 📌 Settings
BASE_PATH = "/Users/dianindrawati/Documents/ERA5_datasets/Runoff/ERA5_q_base_19712000_seasonal.nc"
PRES_PATH = "/Users/dianindrawati/Documents/ERA5_datasets/Runoff/ERA5_q_present_19812010_seasonal.nc"
OUT_CSV = "/Users/dianindrawati/Documents/ERA5_datasets/Runoff/ERA5_q_present_vs_baseline_with_std_land.csv"
VAR = "ro"     # total runoff (m/day → converted to mm/day)
SCALE = 1000   # m/day → mm/day
# Daily files: 1 Dec 1970 – 30 Nov 2000 (baseline), 1 Dec 1980 – 30 Nov 2010 (present);
# lat −90→90 (ascending), lon 0–359. This gives 30 complete seasons per period:
# DJF 1971–2000 / 1981–2010, and MAM/JJA/SON 1971–2000 / 1981–2010
# (1970/1980 contain only December, so the 1970/1980 start adds no extra year).
BASE_YEARS = {"DJF": (1971, 2000), "other": (1970, 2000)}
PRES_YEARS = {"DJF": (1981, 2010), "other": (1980, 2010)}
SEASONS = ["DJF", "MAM", "JJA", "SON"]

# --- Regions ---
regions = {
    "Global": {"lat_min": -60, "lat_max": 60},
    "Tropics": {"lat_min": -23.5, "lat_max": 23.5},
    "Subtropics_N": {"lat_min": 23.5, "lat_max": 35},
    "Subtropics_S": {"lat_min": -35, "lat_max": -23.5},
    "Mid_Latitudes_N": {"lat_min": 35, "lat_max": 60},
    "Mid_Latitudes_S": {"lat_min": -60, "lat_max": -35},
}

SEASON_MONTHS = {"DJF": [12, 1, 2], "MAM": [3, 4, 5], "JJA": [6, 7, 8], "SON": [9, 10, 11]}

# --- Seasonal extraction ---
# DJF is labelled by the January year (Dec 1980 + Jan/Feb 1981 → 1981).
# Seasons missing any of their 3 months are dropped as a safeguard so every
# season is a full 3-month mean (with Dec 1970/1980 in the files, DJF 1971/1981
# is complete and kept).
def extract_season(ds, season):
    time = ds["valid_time"]
    month = time.dt.month
    year = time.dt.year
    season_year = xr.where(month == 12, year + 1, year) if season == "DJF" else year

    ds = ds.assign_coords(season_year=("valid_time", season_year.values))
    ds_season = ds.sel(valid_time=month.isin(SEASON_MONTHS[season]))

    # keep only season-years with all 3 months present
    sy = ds_season["season_year"].values
    mo = ds_season["valid_time"].dt.month.values
    n_months = pd.Series(mo).groupby(sy).nunique()
    complete = n_months[n_months == 3].index.values
    ds_season = ds_season.sel(valid_time=np.isin(sy, complete))

    grouped = ds_season.groupby("season_year").mean(dim="valid_time", skipna=True)
    return grouped.rename({"season_year": "year"})

# --- Load datasets ---
ds_baseline = xr.open_dataset(BASE_PATH)
ds_present = xr.open_dataset(PRES_PATH)

# --- Land mask (Natural Earth 110m, same as annual + CMIP6 analysis) ---
land_regions = regionmask.defined_regions.natural_earth_v5_0_0.land_110
land_mask = land_regions.mask(ds_baseline["lon"], ds_baseline["lat"])
is_land = land_mask.notnull()  # boolean (lat, lon)

# --- Latitude direction ---
latitudes = ds_baseline["lat"].values
lat_descending = latitudes[0] > latitudes[-1]

def sel_lat(da, lat_min, lat_max):
    return da.sel(lat=slice(lat_max, lat_min) if lat_descending else slice(lat_min, lat_max))

# --- Land-only area-weighted mean ---
# Weights = cos(lat) on land, 0 over ocean, applied jointly over (lat, lon)
# so the denominator only counts land area.
def weighted_mean_land(data, land):
    w = np.cos(np.deg2rad(data["lat"])) * land.astype(float)
    w = w.broadcast_like(data.isel(year=0, drop=True))
    return data.weighted(w).mean(dim=("lat", "lon"), skipna=True)

# --- Denominator check ---
# Recomputes the mean by hand as Σ(x·w) / Σ(w over valid land cells) and checks:
#   1) no ocean cell carries any weight,
#   2) no ocean cell has data (so nothing from the ocean reaches the numerator),
#   3) the hand-computed land-only mean equals weighted_mean_land.
# Raises an error if any check fails.
def check_land_denominator(data, land, result, label):
    w = (np.cos(np.deg2rad(data["lat"])) * land.astype(float)).broadcast_like(data.isel(year=0, drop=True))
    valid = data.notnull()
    ocean_weight = float(w.where(~land).sum())
    ocean_valid = int((valid & ~land).sum())
    denom = w.where(valid).sum(dim=("lat", "lon"))                    # land-only Σw per year
    numer = (data * w).sum(dim=("lat", "lon"), skipna=True)
    manual = numer / denom
    max_diff = float(abs(manual - result).max())
    full_band_w = float(np.cos(np.deg2rad(data["lat"])).broadcast_like(w).sum())
    land_frac = float(denom.mean()) / full_band_w                      # share of band area used

    assert ocean_weight == 0.0, f"{label}: ocean cells have weight {ocean_weight}"
    assert ocean_valid == 0, f"{label}: {ocean_valid} ocean cell-years have data"
    assert max_diff < 1e-6, f"{label}: manual vs weighted mean differ by {max_diff}"
    return land_frac

denom_log = []

# --- Compute ---
results = {region: {} for region in regions}

for season in SEASONS:
    key = "DJF" if season == "DJF" else "other"
    base = extract_season(ds_baseline, season).sel(year=slice(*BASE_YEARS[key]))
    present = extract_season(ds_present, season).sel(year=slice(*PRES_YEARS[key]))

    # Unit conversion m/day → mm/day, then land mask
    base_land = (base[VAR] * SCALE).where(is_land)
    present_land = (present[VAR] * SCALE).where(is_land)

    for region, b in regions.items():
        base_reg = sel_lat(base_land, b["lat_min"], b["lat_max"])
        present_reg = sel_lat(present_land, b["lat_min"], b["lat_max"])
        land_reg = sel_lat(is_land, b["lat_min"], b["lat_max"])

        # Spatial land-weighted means per year (mm/day)
        base_sm = weighted_mean_land(base_reg, land_reg)
        present_sm = weighted_mean_land(present_reg, land_reg)

        # Verify the denominator is land-only
        lf_b = check_land_denominator(base_reg, land_reg, base_sm, f"{region} {season} baseline")
        lf_p = check_land_denominator(present_reg, land_reg, present_sm, f"{region} {season} present")
        denom_log.append((region, season, lf_b, lf_p))

        base_mean = base_sm.mean("year").item()
        present_mean = present_sm.mean("year").item()
        base_std = base_sm.std("year").item()
        present_std = present_sm.std("year").item()

        n_base = base_sm.sizes["year"]
        n_present = present_sm.sizes["year"]
        pooled_std = np.sqrt(((n_base - 1) * base_std**2 + (n_present - 1) * present_std**2)
                             / (n_base + n_present - 2))

        diff = present_mean - base_mean

        results[region][season] = {
            "diff_mm_day": diff,
            "pct_change": diff / base_mean * 100,          # relative to baseline land mean
            "std_dev_mm_day": pooled_std,
            "std_dev_percent": pooled_std / base_mean * 100,
            "base_mean_mm_day": base_mean,
            "present_mean_mm_day": present_mean,
            "n_base": n_base,
            "n_present": n_present,
            "n_land_cells": int(land_reg.sum()),
            "n_valid_cells": int(base_reg.isel(year=0).notnull().sum()),  # land cells with ro data
        }

# --- Print denominator verification ---
print("\nDenominator check (passed for every region/season/period):")
print("  ocean weight = 0, ocean data = 0, manual Σ(x·w)/Σ_land(w) == weighted mean")
print(f"{'Region':<20}{'Season':<8}{'Land area frac (base)':>24}{'(present)':>12}")
for region, season, lf_b, lf_p in denom_log:
    print(f"{region:<20}{season:<8}{lf_b:>24.3f}{lf_p:>12.3f}")

# --- Print table ---
print("\nLand-Only Seasonal Runoff Changes (Present - Baseline) with Standard Deviation (cos-lat weighted, land cells only):")
print(f"{'Region':<20}{'Season':<8}{'Diff (mm/day)':>14}{'Change (%)':>12}{'Std Dev (mm/day)':>18}{'Std Dev (%)':>12}{'Land cells (valid)':>22}")
print("-" * 106)
for region in results:
    for season in SEASONS:
        r = results[region][season]
        print(f"{region:<20}{season:<8}{r['diff_mm_day']:>14.6f}{r['pct_change']:>12.2f}"
              f"{r['std_dev_mm_day']:>18.6f}{r['std_dev_percent']:>12.2f}{(str(r['n_land_cells']) + ' (' + str(r['n_valid_cells']) + ')'):>22}")

# --- Detailed statistics ---
print("\n" + "=" * 100)
print("Detailed Land-Only Statistics by Period:")
print("=" * 100)
for region in results:
    print(f"\n{region}:")
    for season in SEASONS:
        r = results[region][season]
        print(f"  {season}: Baseline = {r['base_mean_mm_day']:.6f} mm/day (n={r['n_base']}), "
              f"Present = {r['present_mean_mm_day']:.6f} mm/day (n={r['n_present']}), "
              f"Diff = {r['diff_mm_day']:.6f} mm/day, Pooled Std Dev = {r['std_dev_mm_day']:.6f} mm/day")

# --- Save CSV ---
records = [
    {
        "Region": region,
        "Season": season,
        "Diff_mm_day": m["diff_mm_day"],
        "Pct_Change": m["pct_change"],
        "Std_Dev_mm_day": m["std_dev_mm_day"],
        "Std_Dev_Percent": m["std_dev_percent"],
        "Base_Mean_mm_day": m["base_mean_mm_day"],
        "Present_Mean_mm_day": m["present_mean_mm_day"],
        "N_Base_Years": m["n_base"],
        "N_Present_Years": m["n_present"],
        "N_Land_Cells": m["n_land_cells"],
        "N_Valid_Land_Cells": m["n_valid_cells"],
    }
    for region, seasons_dict in results.items()
    for season, m in seasons_dict.items()
]
pd.DataFrame(records).to_csv(OUT_CSV, index=False)
print(f"\nResults saved to: {OUT_CSV}")

ds_baseline.close()
ds_present.close()


Denominator check (passed for every region/season/period):
  ocean weight = 0, ocean data = 0, manual Σ(x·w)/Σ_land(w) == weighted mean
Region              Season     Land area frac (base)   (present)
Global              DJF                        0.267       0.267
Tropics             DJF                        0.246       0.246
Subtropics_N        DJF                        0.409       0.409
Subtropics_S        DJF                        0.191       0.191
Mid_Latitudes_N     DJF                        0.515       0.515
Mid_Latitudes_S     DJF                        0.034       0.034
Global              MAM                        0.267       0.267
Tropics             MAM                        0.246       0.246
Subtropics_N        MAM                        0.409       0.409
Subtropics_S        MAM                        0.191       0.191
Mid_Latitudes_N     MAM                        0.515       0.515
Mid_Latitudes_S     MAM                        0.034       0.034
Global            